# Embeddings -- Expert

> **Section 01 -- Topic 02 -- expert** | Prerequisites: `advanced.ipynb`

This notebook pushes beyond understanding embeddings into the territory of *engineering* them. We will
dissect the geometry of embedding spaces, probe what information different layers encode, implement
Matryoshka representations that work at any dimensionality, quantize embeddings for production-scale
deployment, and fine-tune models on domain-specific data. Each section contains real, runnable PyTorch
code with matplotlib visualizations so you can see the effects of every technique firsthand.

By the end of this notebook you will be able to diagnose pathological embedding spaces, choose the
right dimensionality-quality tradeoff for your use case, quantize embeddings without destroying recall,
and adapt a pretrained embedding model to a new domain.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print('All imports ready.')

---
## 1. Embedding Space Geometry

### 1.1 Isotropy vs Anisotropy

An *isotropic* embedding space is one where vectors are uniformly distributed across all directions.
In an isotropic space, the average cosine similarity between random pairs of embeddings is close to zero,
which means the full angular range is available for encoding semantic distinctions. This is the ideal
geometry for retrieval: every direction carries information, and cosine similarity is a meaningful
discriminator.

An *anisotropic* space, by contrast, has embeddings clustered in a narrow cone. This is surprisingly
common in transformer-based models. Ethayarajh (2019) showed that BERT, GPT-2, and ELMo all produce
highly anisotropic representations, with average cosine similarity between random sentences exceeding
0.9 in the upper layers. When all vectors point roughly the same direction, cosine similarity loses its
discriminative power -- the difference between a near-duplicate and a completely unrelated sentence
might be 0.95 vs 0.90, making retrieval thresholds fragile.

The root cause is typically a few dominant directions in the embedding matrix that capture frequency
information rather than semantics. During training, high-frequency tokens accumulate large norms and
bias the entire space. Understanding and correcting this phenomenon is the first step toward
production-quality embeddings.

In [ ]:
def measure_isotropy(embeddings: np.ndarray) -> dict:
    """
    Measure isotropy of an embedding space using multiple metrics.
    
    Returns:
        - avg_cosine: average pairwise cosine similarity (lower = more isotropic)
        - partition_function: ratio of min to max eigenvalues of covariance (higher = more isotropic)
        - effective_dimension: how many PCA components capture 90% of variance
    """
    # Average pairwise cosine similarity (sample if large)
    n = len(embeddings)
    if n > 500:
        idx = np.random.choice(n, 500, replace=False)
        sample = embeddings[idx]
    else:
        sample = embeddings
    
    cos_sim = cosine_similarity(sample)
    # Exclude diagonal
    mask = ~np.eye(len(sample), dtype=bool)
    avg_cosine = cos_sim[mask].mean()
    
    # Eigenvalue-based isotropy
    centered = embeddings - embeddings.mean(axis=0)
    cov = np.cov(centered.T)
    eigenvalues = np.linalg.eigvalsh(cov)
    eigenvalues = np.sort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[eigenvalues > 0]
    
    # Partition function ratio (Mu et al., 2018)
    partition = eigenvalues.min() / eigenvalues.max()
    
    # Effective dimensionality (90% variance)
    cumvar = np.cumsum(eigenvalues) / eigenvalues.sum()
    effective_dim = np.searchsorted(cumvar, 0.9) + 1
    
    return {
        'avg_cosine': avg_cosine,
        'partition_ratio': partition,
        'effective_dimension': effective_dim,
        'total_dimension': embeddings.shape[1],
        'eigenvalues': eigenvalues
    }

print('Isotropy measurement function defined.')

In [ ]:
# Generate synthetic isotropic vs anisotropic embeddings for illustration
dim = 128
n_points = 1000

# Isotropic: random unit vectors
iso_raw = np.random.randn(n_points, dim)
isotropic = iso_raw / np.linalg.norm(iso_raw, axis=1, keepdims=True)

# Anisotropic: add a large bias in a few directions
bias = np.zeros(dim)
bias[:3] = 10.0  # first 3 dimensions dominate
aniso_raw = np.random.randn(n_points, dim) * 0.3 + bias
anisotropic = aniso_raw / np.linalg.norm(aniso_raw, axis=1, keepdims=True)

iso_metrics = measure_isotropy(isotropic)
aniso_metrics = measure_isotropy(anisotropic)

print('Isotropic space:')
for k, v in iso_metrics.items():
    if k != 'eigenvalues':
        print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')

print('\nAnisotropic space:')
for k, v in aniso_metrics.items():
    if k != 'eigenvalues':
        print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# Visualize eigenvalue spectra
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(iso_metrics['eigenvalues'][:50], 'o-', color='#2196F3', markersize=4)
axes[0].set_title('Isotropic Space -- Eigenvalue Spectrum', fontsize=13)
axes[0].set_xlabel('Component index')
axes[0].set_ylabel('Eigenvalue (log scale)')
axes[0].axhline(y=iso_metrics['eigenvalues'].mean(), color='red', linestyle='--', alpha=0.7, label='Mean')
axes[0].legend()

axes[1].semilogy(aniso_metrics['eigenvalues'][:50], 'o-', color='#FF5722', markersize=4)
axes[1].set_title('Anisotropic Space -- Eigenvalue Spectrum', fontsize=13)
axes[1].set_xlabel('Component index')
axes[1].set_ylabel('Eigenvalue (log scale)')
axes[1].axhline(y=aniso_metrics['eigenvalues'].mean(), color='red', linestyle='--', alpha=0.7, label='Mean')
axes[1].legend()

plt.tight_layout()
plt.show()
print('Notice the isotropic space has relatively flat eigenvalues while the anisotropic space has a few dominant components.')

### 1.2 Measuring Isotropy of Real Model Embeddings

Let us now measure the isotropy of a real sentence embedding model. We will encode a diverse set of
sentences and examine the geometry of the resulting space. Modern sentence-transformers have been
trained with objectives that encourage more isotropic spaces than raw BERT, but the degree varies
significantly across models.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Diverse sentence set
sentences = [
    'The stock market rallied after the Federal Reserve announcement.',
    'A new species of deep-sea fish was discovered near hydrothermal vents.',
    'The recipe calls for two cups of flour and one egg.',
    'Quantum entanglement enables faster-than-classical communication protocols.',
    'She finished the marathon in under three hours.',
    'The Supreme Court ruling will reshape environmental regulation.',
    'Photosynthesis converts carbon dioxide into glucose using sunlight.',
    'The jazz ensemble performed an improvised set at the downtown club.',
    'Kubernetes orchestrates containerized applications across clusters.',
    'The novel explores themes of identity in post-colonial societies.',
    'Gradient descent minimizes the loss function iteratively.',
    'The architect designed a building inspired by fractal geometry.',
    'Antibiotics should not be prescribed for viral infections.',
    'The trade deficit widened to its largest gap in five years.',
    'Children learn language through a combination of imitation and inference.',
    'The satellite captured high-resolution images of the polar ice caps.',
    'Recursive neural networks process tree-structured data naturally.',
    'The museum exhibit features paintings from the Dutch Golden Age.',
    'Buffer overflow vulnerabilities remain a leading cause of exploits.',
    'The volcanic eruption displaced thousands of residents from their homes.',
] * 5  # repeat to get a reasonable sample size

embeddings = model.encode(sentences, show_progress_bar=False)
real_metrics = measure_isotropy(embeddings)

print(f'Model: all-MiniLM-L6-v2 (dim={embeddings.shape[1]})')
print(f'  Avg pairwise cosine similarity: {real_metrics["avg_cosine"]:.4f}')
print(f'  Partition ratio (min_eig/max_eig): {real_metrics["partition_ratio"]:.6f}')
print(f'  Effective dimension (90% var): {real_metrics["effective_dimension"]} / {real_metrics["total_dimension"]}')

### 1.3 Whitening and Centering Fixes

When an embedding space is anisotropic, we can apply post-hoc corrections to improve its geometry.
The simplest approach is *centering* -- subtracting the mean embedding, which removes the dominant
direction shared by all vectors. A more thorough correction is *whitening*, which centers the data
and then rotates and scales it so the covariance matrix becomes the identity. Whitening transforms
an anisotropic space into a perfectly isotropic one, but can be aggressive -- it amplifies noise in
low-variance directions.

A practical middle ground is *partial whitening* or *ZCA whitening* (zero-phase component analysis),
which whitens while staying as close to the original representation as possible. Su et al. (2021)
showed that whitening BERT embeddings significantly improves performance on semantic textual
similarity benchmarks, sometimes matching or exceeding purpose-built sentence embedding models.

In [ ]:
def whiten_embeddings(embeddings: np.ndarray, method: str = 'full') -> np.ndarray:
    """
    Whiten embeddings to improve isotropy.
    
    Args:
        embeddings: (N, D) array of embeddings
        method: 'center', 'full', or 'partial' (keeps top-k components)
    """
    mean = embeddings.mean(axis=0)
    centered = embeddings - mean
    
    if method == 'center':
        return centered / np.linalg.norm(centered, axis=1, keepdims=True)
    
    # Compute SVD
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    
    if method == 'full':
        # Full whitening: X_white = U (all singular values become 1)
        whitened = U
    elif method == 'partial':
        # Partial: keep top-k components, then whiten
        k = min(128, len(S))
        whitened = U[:, :k]
    else:
        raise ValueError(f'Unknown method: {method}')
    
    return whitened / np.linalg.norm(whitened, axis=1, keepdims=True)

# Apply corrections
centered_emb = whiten_embeddings(embeddings, method='center')
whitened_emb = whiten_embeddings(embeddings, method='full')

centered_metrics = measure_isotropy(centered_emb)
whitened_metrics = measure_isotropy(whitened_emb)

print('After centering:')
print(f'  Avg cosine: {centered_metrics["avg_cosine"]:.4f} (was {real_metrics["avg_cosine"]:.4f})')
print(f'  Effective dim: {centered_metrics["effective_dimension"]} (was {real_metrics["effective_dimension"]})')
print(f'\nAfter full whitening:')
print(f'  Avg cosine: {whitened_metrics["avg_cosine"]:.4f}')
print(f'  Effective dim: {whitened_metrics["effective_dimension"]}')

In [ ]:
# Visualize the effect with PCA projections
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, emb, title in zip(
    axes,
    [embeddings, centered_emb, whitened_emb],
    ['Original', 'Centered', 'Whitened']
):
    pca = PCA(n_components=2)
    proj = pca.fit_transform(emb)
    ax.scatter(proj[:, 0], proj[:, 1], alpha=0.5, s=20, c='#3F51B5')
    ax.set_title(f'{title}\navg cos={measure_isotropy(emb)["avg_cosine"]:.3f}', fontsize=12)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_aspect('equal')

plt.suptitle('Effect of Whitening on Embedding Geometry (PCA 2D)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Probing Embeddings

### 2.1 What Information Is Encoded?

Embeddings are dense vectors, but what exactly do they capture? The *probing* methodology provides
a principled way to answer this question. The idea is simple: freeze the embeddings and train a
lightweight classifier (typically a linear model) to predict a linguistic property from the embedding.
If the classifier succeeds, the information was present in the embedding; if it fails, it was not.

Probing has revealed a rich picture of what different layers encode. Lower layers of transformers
tend to capture surface-level features -- part-of-speech tags, word identity, and character-level
patterns. Middle layers encode syntactic structure -- dependency relations, constituency parse
information, and named entity types. Upper layers capture more semantic and task-specific information
-- sentiment, semantic roles, and discourse relations.

This layered encoding has practical implications. If you are building a POS tagger, embeddings from
lower layers may be more useful than the final layer. If you are building a semantic search system,
the final layer is likely best. Understanding what each layer encodes helps you make informed
decisions about which representations to use for downstream tasks.

A critical subtlety: probing accuracy must be compared against baselines. A probe trained on random
embeddings of the same dimensionality provides a lower bound. If your probe barely beats this baseline,
the information may not be meaningfully encoded -- the probe might just be exploiting statistical
regularities in the data.

In [ ]:
# Create synthetic probing dataset
# We simulate embeddings that encode different properties at different "layers"

def create_probing_data(n_samples=2000, dim=256):
    """
    Create synthetic data mimicking how different layers encode different properties.
    Layer 0 (early): encodes POS-like features strongly
    Layer 1 (mid): encodes NER-like features strongly  
    Layer 2 (late): encodes sentiment strongly
    """
    np.random.seed(SEED)
    
    # Labels
    pos_labels = np.random.randint(0, 5, n_samples)    # 5 POS tags
    ner_labels = np.random.randint(0, 4, n_samples)    # 4 NER types
    sent_labels = np.random.randint(0, 3, n_samples)   # 3 sentiments
    
    layers = {}
    for layer_idx in range(3):
        base = np.random.randn(n_samples, dim) * 0.1
        
        # Each layer encodes one property more strongly
        if layer_idx == 0:  # Early layer -- POS
            for i, label in enumerate(pos_labels):
                base[i, label*10:(label+1)*10] += 2.0
            for i, label in enumerate(ner_labels):
                base[i, 50 + label*5:50 + (label+1)*5] += 0.3
            for i, label in enumerate(sent_labels):
                base[i, 80 + label*3:80 + (label+1)*3] += 0.1
                
        elif layer_idx == 1:  # Middle layer -- NER
            for i, label in enumerate(pos_labels):
                base[i, label*10:(label+1)*10] += 0.8
            for i, label in enumerate(ner_labels):
                base[i, 50 + label*10:50 + (label+1)*10] += 2.0
            for i, label in enumerate(sent_labels):
                base[i, 100 + label*5:100 + (label+1)*5] += 0.5
                
        else:  # Late layer -- Sentiment
            for i, label in enumerate(pos_labels):
                base[i, label*5:(label+1)*5] += 0.3
            for i, label in enumerate(ner_labels):
                base[i, 50 + label*5:50 + (label+1)*5] += 0.6
            for i, label in enumerate(sent_labels):
                base[i, 100 + label*15:100 + (label+1)*15] += 2.5
        
        layers[layer_idx] = base
    
    return layers, pos_labels, ner_labels, sent_labels

layers, pos_labels, ner_labels, sent_labels = create_probing_data()
print(f'Created probing dataset: {len(pos_labels)} samples, 3 layers, dim={layers[0].shape[1]}')
print(f'Tasks: POS ({len(np.unique(pos_labels))} classes), NER ({len(np.unique(ner_labels))} classes), '
      f'Sentiment ({len(np.unique(sent_labels))} classes)')

In [ ]:
# Train linear probes for each property at each layer
tasks = {
    'POS': pos_labels,
    'NER': ner_labels,
    'Sentiment': sent_labels,
}

layer_names = ['Early (Layer 0)', 'Middle (Layer 1)', 'Late (Layer 2)']
results = {}

for task_name, labels in tasks.items():
    results[task_name] = []
    for layer_idx in range(3):
        X = layers[layer_idx]
        X_train, X_test, y_train, y_test = train_test_split(
            X, labels, test_size=0.3, random_state=SEED
        )
        
        probe = LogisticRegression(max_iter=1000, random_state=SEED)
        probe.fit(X_train, y_train)
        acc = probe.score(X_test, y_test)
        results[task_name].append(acc)

# Print results
print(f'{"Task":<12} {"Early":>10} {"Middle":>10} {"Late":>10}')
print('-' * 45)
for task_name, accs in results.items():
    best_idx = np.argmax(accs)
    row = f'{task_name:<12}'
    for i, acc in enumerate(accs):
        marker = ' *' if i == best_idx else '  '
        row += f' {acc:>8.1%}{marker}'
    print(row)
print('\n* = best layer for this task')

In [ ]:
# Visualize probing results
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(3)
width = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']

for i, (task_name, accs) in enumerate(results.items()):
    bars = ax.bar(x + i * width, accs, width, label=task_name, color=colors[i], alpha=0.85)

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Probe Accuracy', fontsize=12)
ax.set_title('Linear Probe Accuracy by Layer and Task', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(layer_names)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)

# Add random baseline
for task_name, labels in tasks.items():
    baseline = 1.0 / len(np.unique(labels))
    ax.axhline(y=baseline, color='gray', linestyle=':', alpha=0.4)

ax.text(2.6, 0.22, 'Random\nbaselines', fontsize=9, color='gray', alpha=0.7)

plt.tight_layout()
plt.show()
print('Each task peaks at the layer that most strongly encodes its corresponding information.')

### 2.2 Comparing Information Across Models

The probing framework is also invaluable for comparing models. If you are choosing between two
embedding models for a named-entity-recognition-augmented retrieval system, probing both models for
NER accuracy tells you which one preserves more entity information. This is a far more informative
comparison than looking at generic benchmarks like STS (semantic textual similarity), which may not
reflect your specific needs.

Below we compare random embeddings (our lower-bound baseline) against the synthetic "late layer"
embeddings to confirm that our probes are detecting real signal rather than artifacts.

In [ ]:
# Baseline: random embeddings should perform near chance
random_emb = np.random.randn(len(pos_labels), 256)

print('Random embedding baseline:')
for task_name, labels in tasks.items():
    X_train, X_test, y_train, y_test = train_test_split(
        random_emb, labels, test_size=0.3, random_state=SEED
    )
    probe = LogisticRegression(max_iter=1000, random_state=SEED)
    probe.fit(X_train, y_train)
    acc = probe.score(X_test, y_test)
    chance = 1.0 / len(np.unique(labels))
    print(f'  {task_name}: {acc:.1%} (chance = {chance:.1%})')

print('\nRandom embeddings perform near chance, confirming probes detect real encoded information.')

---
## 3. Matryoshka Embeddings

### 3.1 The Core Idea

Matryoshka Representation Learning (MRL), introduced by Kusupati et al. (2022), trains a single
embedding model whose representations are useful at *any* prefix dimensionality. If your model
produces 768-dimensional embeddings, a Matryoshka model ensures that the first 64 dimensions, the
first 128, the first 256, and so on, all form valid embeddings with gracefully degrading quality.

The name comes from Russian nesting dolls -- each smaller representation is "nested" inside the
larger one. This is achieved by a deceptively simple training trick: during training, the loss
function is evaluated not just on the full embedding but also on truncated prefixes. The model
is forced to pack the most important information into the earliest dimensions.

The practical benefits are enormous. At inference time, you can choose your dimensionality based on
your latency/quality budget. You can use short embeddings for coarse filtering and full embeddings
for reranking. You can store full embeddings in cold storage and truncated versions in a fast
in-memory index. All of this from a single model checkpoint, with no retraining needed.

The quality degradation profile is remarkably gentle. Kusupati et al. showed that truncating to
1/16th of the full dimension typically costs less than 5% accuracy on standard benchmarks, while
reducing storage and search time by 16x.

In [ ]:
class MatryoshkaLoss(nn.Module):
    """
    Matryoshka Representation Learning loss.
    Computes contrastive loss at multiple truncation dimensions,
    weighting earlier (smaller) truncations more heavily.
    """
    def __init__(self, full_dim: int, truncation_dims: list[int], temperature: float = 0.05):
        super().__init__()
        self.truncation_dims = sorted(truncation_dims)
        self.temperature = temperature
        # Log-uniform weights: smaller dims get higher relative weight
        self.weights = [1.0 / np.log2(d) for d in self.truncation_dims]
        total = sum(self.weights)
        self.weights = [w / total for w in self.weights]
        
    def contrastive_loss(self, emb_a: torch.Tensor, emb_b: torch.Tensor) -> torch.Tensor:
        """InfoNCE loss: each (a_i, b_i) is a positive pair."""
        emb_a = F.normalize(emb_a, dim=-1)
        emb_b = F.normalize(emb_b, dim=-1)
        logits = emb_a @ emb_b.T / self.temperature
        labels = torch.arange(len(emb_a), device=emb_a.device)
        return F.cross_entropy(logits, labels)
    
    def forward(self, emb_a: torch.Tensor, emb_b: torch.Tensor) -> torch.Tensor:
        total_loss = 0.0
        for dim, weight in zip(self.truncation_dims, self.weights):
            loss_at_dim = self.contrastive_loss(emb_a[:, :dim], emb_b[:, :dim])
            total_loss += weight * loss_at_dim
        return total_loss

# Demonstrate
mrl_loss = MatryoshkaLoss(
    full_dim=256,
    truncation_dims=[32, 64, 128, 256]
)
print('Matryoshka loss truncation dims and weights:')
for d, w in zip(mrl_loss.truncation_dims, mrl_loss.weights):
    print(f'  dim={d:>4d}  weight={w:.4f}')

In [ ]:
# Train a simple embedding model with vs without Matryoshka loss

class SimpleEmbedder(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=256, hidden_dim=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.proj = nn.Linear(hidden_dim, embed_dim)
    
    def forward(self, x):
        # Mean pool over sequence
        return self.proj(self.embed(x).mean(dim=1))

# Generate synthetic training pairs
vocab_size = 1000
seq_len = 10
n_pairs = 2000
batch_size = 128

anchor_ids = torch.randint(0, vocab_size, (n_pairs, seq_len))
# Positive: perturbed version (swap ~20% of tokens)
positive_ids = anchor_ids.clone()
mask = torch.rand(n_pairs, seq_len) < 0.2
positive_ids[mask] = torch.randint(0, vocab_size, (mask.sum().item(),))

# Train standard model
model_std = SimpleEmbedder(vocab_size, 256)
opt_std = torch.optim.Adam(model_std.parameters(), lr=1e-3)
std_loss_fn = MatryoshkaLoss(256, [256])  # Only full dim

# Train Matryoshka model
model_mrl = SimpleEmbedder(vocab_size, 256)
opt_mrl = torch.optim.Adam(model_mrl.parameters(), lr=1e-3)
mrl_loss_fn = MatryoshkaLoss(256, [32, 64, 128, 256])

epochs = 15
std_losses, mrl_losses = [], []

for epoch in range(epochs):
    perm = torch.randperm(n_pairs)
    epoch_std_loss = 0
    epoch_mrl_loss = 0
    n_batches = 0
    
    for i in range(0, n_pairs, batch_size):
        idx = perm[i:i+batch_size]
        a, p = anchor_ids[idx], positive_ids[idx]
        
        # Standard
        opt_std.zero_grad()
        emb_a, emb_p = model_std(a), model_std(p)
        loss_std = std_loss_fn(emb_a, emb_p)
        loss_std.backward()
        opt_std.step()
        epoch_std_loss += loss_std.item()
        
        # Matryoshka
        opt_mrl.zero_grad()
        emb_a, emb_p = model_mrl(a), model_mrl(p)
        loss_mrl = mrl_loss_fn(emb_a, emb_p)
        loss_mrl.backward()
        opt_mrl.step()
        epoch_mrl_loss += loss_mrl.item()
        
        n_batches += 1
    
    std_losses.append(epoch_std_loss / n_batches)
    mrl_losses.append(epoch_mrl_loss / n_batches)

print(f'Training complete. Final losses -- Standard: {std_losses[-1]:.4f}, Matryoshka: {mrl_losses[-1]:.4f}')

In [ ]:
# Benchmark quality at different truncation points
def retrieval_accuracy(model, anchors, positives, dim):
    """Compute retrieval@1 accuracy at a given embedding dimension."""
    with torch.no_grad():
        emb_a = F.normalize(model(anchors)[:, :dim], dim=-1)
        emb_p = F.normalize(model(positives)[:, :dim], dim=-1)
    
    sims = (emb_a @ emb_p.T).numpy()
    predictions = sims.argmax(axis=1)
    targets = np.arange(len(anchors))
    return (predictions == targets).mean()

# Test set
test_n = 200
test_anchors = torch.randint(0, vocab_size, (test_n, seq_len))
test_positives = test_anchors.clone()
test_mask = torch.rand(test_n, seq_len) < 0.2
test_positives[test_mask] = torch.randint(0, vocab_size, (test_mask.sum().item(),))

dims = [16, 32, 64, 128, 256]
std_accs = [retrieval_accuracy(model_std, test_anchors, test_positives, d) for d in dims]
mrl_accs = [retrieval_accuracy(model_mrl, test_anchors, test_positives, d) for d in dims]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(dims, std_accs, 'o-', color='#F44336', linewidth=2, markersize=8, label='Standard (full-dim only)')
ax.plot(dims, mrl_accs, 's-', color='#2196F3', linewidth=2, markersize=8, label='Matryoshka (multi-dim)')
ax.set_xlabel('Embedding Dimension (truncation point)', fontsize=12)
ax.set_ylabel('Retrieval@1 Accuracy', fontsize=12)
ax.set_title('Matryoshka vs Standard: Quality at Different Dimensions', fontsize=14)
ax.set_xscale('log', base=2)
ax.set_xticks(dims)
ax.set_xticklabels(dims)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('The Matryoshka model maintains quality at lower dimensions where the standard model degrades sharply.')

---
## 4. Binary and Scalar Quantization

### 4.1 The Storage-Quality Tradeoff

In production retrieval systems, embedding storage and comparison speed are often the bottleneck.
A 768-dimensional float32 embedding consumes 3,072 bytes. At a billion documents, that is 3 TB of
embedding storage alone -- not counting the index overhead. Quantization compresses embeddings by
reducing the precision of each dimension.

There are two main approaches. **Scalar quantization** maps each float32 value to a lower-precision
integer. The most common variant maps to int8 (256 levels), reducing storage by 4x. The mapping
preserves the relative ordering within each dimension, so cosine similarity is approximately
preserved. The quality loss is typically negligible -- less than 1% recall drop for most models.

**Binary quantization** is more aggressive: each dimension becomes a single bit -- positive or
negative. This yields 32x compression (float32 to 1 bit) and enables ultra-fast comparison using
hardware-accelerated Hamming distance (popcount instructions). The quality loss is more significant,
typically 5-15% recall depending on the model and dataset. However, binary embeddings are
extraordinary for the first stage of a multi-stage retrieval pipeline, where you need to quickly
narrow billions of candidates to millions before applying more expensive reranking.

The choice depends on your latency budget, scale, and whether you can afford a reranking stage.
Many production systems use a tiered approach: binary quantization for the broadest filter,
int8 for the intermediate stage, and full float32 only for the final reranking.

In [ ]:
def scalar_quantize(embeddings: np.ndarray, bits: int = 8) -> tuple:
    """
    Scalar quantization: map each dimension to uint8.
    Returns quantized array and calibration parameters for dequantization.
    """
    levels = 2 ** bits - 1
    
    # Per-dimension min/max calibration
    mins = embeddings.min(axis=0)
    maxs = embeddings.max(axis=0)
    ranges = maxs - mins
    ranges[ranges == 0] = 1.0  # avoid division by zero
    
    # Normalize to [0, 1] then scale to [0, levels]
    normalized = (embeddings - mins) / ranges
    quantized = np.round(normalized * levels).astype(np.uint8)
    
    return quantized, mins, ranges


def scalar_dequantize(quantized: np.ndarray, mins: np.ndarray, ranges: np.ndarray, bits: int = 8) -> np.ndarray:
    """Reconstruct float embeddings from quantized values."""
    levels = 2 ** bits - 1
    return (quantized.astype(np.float32) / levels) * ranges + mins


def binary_quantize(embeddings: np.ndarray) -> np.ndarray:
    """
    Binary quantization: each dimension becomes 1 bit.
    Returns packed binary representation.
    """
    binary = (embeddings > 0).astype(np.uint8)
    # Pack bits for efficient storage
    n_samples, dim = binary.shape
    # Pad to multiple of 8
    pad_dim = int(np.ceil(dim / 8) * 8)
    if pad_dim > dim:
        binary = np.pad(binary, ((0, 0), (0, pad_dim - dim)))
    packed = np.packbits(binary, axis=1)
    return packed


def hamming_similarity(packed_a: np.ndarray, packed_b: np.ndarray) -> np.ndarray:
    """Compute similarity from Hamming distance on packed binary embeddings."""
    n_bits = packed_a.shape[1] * 8
    # XOR and count differing bits
    xor = np.bitwise_xor(packed_a[:, np.newaxis, :], packed_b[np.newaxis, :, :])
    hamming_dist = np.unpackbits(xor, axis=-1).sum(axis=-1)
    # Convert to similarity: 1 - (dist / n_bits)
    return 1.0 - hamming_dist / n_bits

print('Quantization functions defined.')

In [ ]:
# Encode real sentences and benchmark quantization
test_sentences = [
    'Machine learning models require large amounts of training data.',
    'Deep learning uses neural networks with many layers.',
    'Natural language processing analyzes human language.',
    'Computer vision enables machines to interpret images.',
    'Reinforcement learning trains agents through rewards.',
    'Transfer learning leverages pretrained models for new tasks.',
    'The cat sat on the mat and watched the birds.',
    'She drove to the grocery store to buy vegetables.',
    'The football match ended in a dramatic penalty shootout.',
    'Climate change poses significant risks to coastal communities.',
]

query_sentences = [
    'Neural networks need lots of data for training.',     # Similar to 0
    'Image recognition by artificial intelligence systems.',  # Similar to 3
    'Global warming threatens seaside populations.',          # Similar to 9
]

doc_emb = model.encode(test_sentences)
query_emb = model.encode(query_sentences)

# Full precision cosine similarity (ground truth)
full_sims = cosine_similarity(query_emb, doc_emb)

# Scalar quantized similarity
doc_q, mins, ranges = scalar_quantize(doc_emb)
query_q, _, _ = scalar_quantize(query_emb)  # Use same calibration ideally, simplified here
doc_deq = scalar_dequantize(doc_q, mins, ranges)
query_deq = scalar_dequantize(query_q, mins, ranges)
int8_sims = cosine_similarity(query_deq, doc_deq)

# Binary quantized similarity
doc_bin = binary_quantize(doc_emb)
query_bin = binary_quantize(query_emb)
bin_sims = hamming_similarity(query_bin, doc_bin)

print('Query: "Neural networks need lots of data for training."')
print(f'{"Document":<55} {"FP32":>6} {"INT8":>6} {"Binary":>6}')
print('-' * 80)
for i, sent in enumerate(test_sentences):
    print(f'{sent[:53]:<55} {full_sims[0][i]:>6.3f} {int8_sims[0][i]:>6.3f} {bin_sims[0][i]:>6.3f}')

In [ ]:
# Benchmark: recall@k across quantization methods
def recall_at_k(query_sims, gt_sims, k=3):
    """What fraction of true top-k results appear in the quantized top-k?"""
    recalls = []
    for q_idx in range(len(query_sims)):
        gt_topk = set(np.argsort(gt_sims[q_idx])[-k:])
        pred_topk = set(np.argsort(query_sims[q_idx])[-k:])
        recalls.append(len(gt_topk & pred_topk) / k)
    return np.mean(recalls)

for k in [1, 3, 5]:
    r_int8 = recall_at_k(int8_sims, full_sims, k)
    r_bin = recall_at_k(bin_sims, full_sims, k)
    print(f'Recall@{k}: INT8={r_int8:.1%}  Binary={r_bin:.1%}')

# Storage comparison
dim = doc_emb.shape[1]
n_docs = 1_000_000
fp32_bytes = n_docs * dim * 4
int8_bytes = n_docs * dim * 1
binary_bytes = n_docs * (dim // 8)

print(f'\nStorage for {n_docs:,} documents ({dim}d):')
print(f'  FP32:   {fp32_bytes / 1e9:.2f} GB')
print(f'  INT8:   {int8_bytes / 1e9:.2f} GB ({fp32_bytes / int8_bytes:.0f}x compression)')
print(f'  Binary: {binary_bytes / 1e9:.2f} GB ({fp32_bytes / binary_bytes:.0f}x compression)')

In [ ]:
# Visualize the storage-quality tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: similarity correlation
methods = ['INT8 Scalar', 'Binary']
sim_sets = [int8_sims.flatten(), bin_sims.flatten()]
colors = ['#4CAF50', '#FF9800']

for method, sims, color in zip(methods, sim_sets, colors):
    ax1.scatter(full_sims.flatten(), sims, alpha=0.6, s=40, label=method, color=color)

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect correlation')
ax1.set_xlabel('FP32 Cosine Similarity', fontsize=11)
ax1.set_ylabel('Quantized Similarity', fontsize=11)
ax1.set_title('Similarity Preservation Under Quantization', fontsize=13)
ax1.legend(fontsize=10)

# Right: storage vs quality bar chart
storage_gb = [fp32_bytes/1e9, int8_bytes/1e9, binary_bytes/1e9]
bar_labels = ['FP32', 'INT8', 'Binary']
bar_colors = ['#2196F3', '#4CAF50', '#FF9800']

bars = ax2.bar(bar_labels, storage_gb, color=bar_colors, alpha=0.85)
ax2.set_ylabel('Storage (GB) for 1M docs', fontsize=11)
ax2.set_title('Storage Requirements by Precision', fontsize=13)

for bar, gb in zip(bars, storage_gb):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{gb:.2f} GB', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 5. Fine-Tuning Embeddings

### 5.1 Why Fine-Tune?

General-purpose embedding models are trained on broad web data and perform well on standard benchmarks.
But in domain-specific settings -- legal documents, medical records, codebases, internal company
knowledge -- they often underperform because the vocabulary, writing style, and concept of "similarity"
differ from their training distribution.

Fine-tuning adapts a pretrained embedding model to your domain by training on domain-specific
positive pairs. The most effective modern approach is **Multiple Negatives Ranking Loss (MNRL)**,
which treats all other examples in a batch as negatives. Given a batch of (query, positive) pairs,
MNRL computes a cross-entropy loss where the correct positive should have higher similarity than
all other positives in the batch. This is identical to the InfoNCE loss we implemented above.

MNRL is efficient because it requires only positive pairs -- no explicit negatives. The batch itself
provides hard negatives (other positives that happen to be semantically close to your query). Larger
batch sizes yield harder negatives and better training signal.

Other loss functions have different strengths. **Contrastive loss** operates on pairs with explicit
positive/negative labels and a margin hyperparameter. **Triplet loss** takes (anchor, positive,
negative) triplets and pushes the positive closer than the negative by a margin. Both require
careful negative mining, which makes them more complex to use but potentially more precise when
you have curated hard negatives.

In [ ]:
# Implement contrastive, triplet, and MNRL losses from scratch in PyTorch

class ContrastiveLoss(nn.Module):
    """Pair-based contrastive loss with margin."""
    def __init__(self, margin: float = 0.5):
        super().__init__()
        self.margin = margin
    
    def forward(self, emb_a: torch.Tensor, emb_b: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """labels: 1 for positive pairs, 0 for negative pairs."""
        distances = 1 - F.cosine_similarity(emb_a, emb_b)
        pos_loss = labels * distances.pow(2)
        neg_loss = (1 - labels) * F.relu(self.margin - distances).pow(2)
        return (pos_loss + neg_loss).mean()


class TripletLoss(nn.Module):
    """Triplet loss with margin."""
    def __init__(self, margin: float = 0.3):
        super().__init__()
        self.margin = margin
    
    def forward(self, anchor: torch.Tensor, positive: torch.Tensor, negative: torch.Tensor) -> torch.Tensor:
        pos_dist = 1 - F.cosine_similarity(anchor, positive)
        neg_dist = 1 - F.cosine_similarity(anchor, negative)
        return F.relu(pos_dist - neg_dist + self.margin).mean()


class MNRLoss(nn.Module):
    """Multiple Negatives Ranking Loss (InfoNCE)."""
    def __init__(self, temperature: float = 0.05):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, query: torch.Tensor, positive: torch.Tensor) -> torch.Tensor:
        query = F.normalize(query, dim=-1)
        positive = F.normalize(positive, dim=-1)
        logits = query @ positive.T / self.temperature
        labels = torch.arange(len(query), device=query.device)
        return F.cross_entropy(logits, labels)


# Quick sanity check
batch = 16
dim = 128
a = torch.randn(batch, dim)
b = torch.randn(batch, dim)
c = torch.randn(batch, dim)

print(f'Contrastive loss: {ContrastiveLoss()(a, b, torch.ones(batch)):.4f}')
print(f'Triplet loss:     {TripletLoss()(a, b, c):.4f}')
print(f'MNRL loss:        {MNRLoss()(a, b):.4f}')

### 5.2 Creating Domain-Specific Training Data

The quality of fine-tuning depends almost entirely on the quality of your training pairs. For
retrieval applications, each pair should represent a (query, relevant document) relationship.
Here are proven strategies for generating training data:

1. **Click-through logs**: If you have a search system, user clicks provide natural positive pairs.
2. **QA pairs**: Questions paired with their answer passages.
3. **Title-abstract pairs**: Paper titles paired with their abstracts.
4. **Synthetic generation**: Use an LLM to generate questions for your documents.
5. **Cross-encoder distillation**: Use a cross-encoder to score pairs and keep the top-scoring ones.

Below we create a synthetic technical documentation dataset and fine-tune a sentence-transformer.

In [ ]:
# Create synthetic domain-specific training pairs (technical documentation)
training_pairs = [
    ('How do I configure SSL certificates?', 'To set up SSL/TLS, generate a certificate signing request using openssl and submit it to your CA.'),
    ('Database connection pool settings', 'Configure max_connections, idle_timeout, and pool_size in your database.yml configuration file.'),
    ('API rate limiting configuration', 'Rate limits can be set per endpoint using the rate_limit directive with requests per minute.'),
    ('How to enable debug logging', 'Set the LOG_LEVEL environment variable to DEBUG or update the logging.conf file.'),
    ('Memory allocation tuning', 'Adjust heap_size_mb and stack_size_kb parameters in the runtime configuration.'),
    ('Load balancer health checks', 'Configure health_check_path and health_check_interval in your load balancer settings.'),
    ('Authentication token expiry', 'JWT tokens expire after the duration specified in token_ttl_seconds, default 3600.'),
    ('Backup and disaster recovery', 'Automated backups run daily at the time specified in backup_schedule with retention_days.'),
    ('Container orchestration scaling', 'Set min_replicas and max_replicas in the autoscaler configuration with target CPU threshold.'),
    ('Network firewall rules setup', 'Define ingress and egress rules in the firewall.rules.json file with CIDR notation.'),
    ('Cache invalidation strategy', 'Use TTL-based expiration or event-driven invalidation via the cache.invalidate webhook.'),
    ('Monitoring and alerting setup', 'Configure alert thresholds in monitoring.yml with notification channels for each severity level.'),
    ('Service mesh configuration', 'The sidecar proxy intercepts traffic based on rules in the mesh-config ConfigMap.'),
    ('Data pipeline scheduling', 'DAGs are scheduled using cron expressions in the pipeline_schedule field of each workflow.'),
    ('Feature flag management', 'Toggle features per environment using the feature_flags table with rollout_percentage.'),
    ('Secret management best practices', 'Store secrets in a vault service and reference them via environment variable injection.'),
    ('CDN cache configuration', 'Set Cache-Control headers and configure edge cache TTL in your CDN provider dashboard.'),
    ('Microservice communication patterns', 'Use synchronous REST for queries and asynchronous message queues for commands.'),
    ('Database migration strategy', 'Run migrations with the migrate command, ensuring backward compatibility for zero-downtime deploys.'),
    ('Logging aggregation setup', 'Ship logs from each service to the central collector using the fluentd sidecar pattern.'),
]

# Convert to InputExample format
train_examples = [
    InputExample(texts=[q, a]) for q, a in training_pairs
]

print(f'Created {len(train_examples)} training pairs for fine-tuning.')
print(f'Example: "{training_pairs[0][0]}" -> "{training_pairs[0][1][:60]}..."')

In [ ]:
# Evaluate base model before fine-tuning
base_model = SentenceTransformer('all-MiniLM-L6-v2')

eval_queries = [
    'How do I set up HTTPS?',
    'What are the database pool options?',
    'How to configure API throttling?',
    'Where do I change the log verbosity?',
    'How to adjust memory settings?',
]

eval_docs = [pair[1] for pair in training_pairs[:10]]

def evaluate_retrieval(model, queries, docs, name='Model'):
    q_emb = model.encode(queries)
    d_emb = model.encode(docs)
    sims = cosine_similarity(q_emb, d_emb)
    
    print(f'\n{name} -- Retrieval Results:')
    correct = 0
    for i, query in enumerate(queries):
        top_idx = sims[i].argmax()
        is_correct = top_idx == i
        correct += is_correct
        marker = 'OK' if is_correct else 'MISS'
        print(f'  [{marker}] "{query[:45]}" -> doc {top_idx} (sim={sims[i][top_idx]:.3f})')
    
    print(f'  Accuracy: {correct}/{len(queries)} = {correct/len(queries):.0%}')
    return sims

base_sims = evaluate_retrieval(base_model, eval_queries, eval_docs, 'Base Model')

In [ ]:
# Fine-tune with MNRL loss
ft_model = SentenceTransformer('all-MiniLM-L6-v2')

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
train_loss = losses.MultipleNegativesRankingLoss(ft_model)

# Fine-tune for a few epochs
ft_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10,
    warmup_steps=10,
    show_progress_bar=True,
    output_path=None,
)

print('Fine-tuning complete.')

In [ ]:
# Evaluate fine-tuned model
ft_sims = evaluate_retrieval(ft_model, eval_queries, eval_docs, 'Fine-Tuned Model')

In [ ]:
# Visualize before vs after fine-tuning
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, sims, title in zip(axes, [base_sims, ft_sims], ['Base Model', 'Fine-Tuned Model']):
    im = ax.imshow(sims, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_title(f'{title} -- Query-Document Similarity', fontsize=12)
    ax.set_xlabel('Document Index')
    ax.set_ylabel('Query Index')
    
    # Annotate cells
    for i in range(sims.shape[0]):
        for j in range(sims.shape[1]):
            color = 'white' if sims[i, j] > 0.6 else 'black'
            ax.text(j, i, f'{sims[i,j]:.2f}', ha='center', va='center', fontsize=8, color=color)
    
    # Highlight diagonal (correct matches)
    for i in range(min(sims.shape)):
        rect = plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, edgecolor='red', linewidth=2)
        ax.add_patch(rect)

plt.colorbar(im, ax=axes, shrink=0.8, label='Cosine Similarity')
plt.tight_layout()
plt.show()
print('Red boxes indicate correct query-document pairs. Fine-tuning should strengthen the diagonal.')

In [ ]:
# Visualize embedding space with t-SNE: base vs fine-tuned
from sklearn.manifold import TSNE

all_texts = [q for q, _ in training_pairs] + [a for _, a in training_pairs]
labels = ['query'] * len(training_pairs) + ['document'] * len(training_pairs)
pair_ids = list(range(len(training_pairs))) * 2

base_emb = base_model.encode(all_texts)
ft_emb = ft_model.encode(all_texts)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, emb, title in zip(axes, [base_emb, ft_emb], ['Base Model', 'Fine-Tuned Model']):
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=10)
    proj = tsne.fit_transform(emb)
    
    n_pairs = len(training_pairs)
    q_proj = proj[:n_pairs]
    d_proj = proj[n_pairs:]
    
    # Draw lines connecting query-document pairs
    for i in range(n_pairs):
        ax.plot([q_proj[i, 0], d_proj[i, 0]], [q_proj[i, 1], d_proj[i, 1]],
                'k-', alpha=0.15, linewidth=0.8)
    
    ax.scatter(q_proj[:, 0], q_proj[:, 1], c='#2196F3', s=50, label='Queries', zorder=5, edgecolors='white', linewidth=0.5)
    ax.scatter(d_proj[:, 0], d_proj[:, 1], c='#FF5722', s=50, label='Documents', zorder=5, edgecolors='white', linewidth=0.5)
    ax.set_title(f'{title} -- Query-Doc Embedding Space', fontsize=13)
    ax.legend(fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()
print('After fine-tuning, matching query-document pairs should cluster closer together.')

### 5.3 Evaluation: Is Fine-Tuning Worth It?

Fine-tuning is not always beneficial. If your domain is well-represented in the pretraining data,
or if your training pairs are noisy, fine-tuning can actually hurt performance. Always evaluate
rigorously on a held-out test set that reflects your actual use case.

Key evaluation dimensions:
- **Retrieval accuracy** (Recall@k, MRR, NDCG): Does the model find the right documents?
- **Semantic textual similarity**: Does the model's notion of similarity align with human judgments?
- **Domain specificity vs generality**: Has the model forgotten general knowledge? Test on both
  domain-specific and general benchmarks.
- **Embedding space quality**: Check isotropy metrics before and after -- fine-tuning can sometimes
  make the space more anisotropic.

In [ ]:
# Compare isotropy before and after fine-tuning
base_iso = measure_isotropy(base_emb)
ft_iso = measure_isotropy(ft_emb)

print('Embedding Space Health Check:')
print(f'{"Metric":<30} {"Base":>10} {"Fine-Tuned":>12}')
print('-' * 55)
for key in ['avg_cosine', 'partition_ratio', 'effective_dimension']:
    bv = base_iso[key]
    fv = ft_iso[key]
    if isinstance(bv, float):
        print(f'{key:<30} {bv:>10.4f} {fv:>12.4f}')
    else:
        print(f'{key:<30} {bv:>10} {fv:>12}')

---
## Summary

In this expert notebook we covered five advanced embedding topics:

1. **Embedding Space Geometry**: Isotropic spaces enable better retrieval. We measured isotropy using
   eigenvalue analysis and average cosine similarity, then applied whitening to correct anisotropy.

2. **Probing Embeddings**: Linear probes reveal what information each layer encodes. Lower layers
   capture syntax, upper layers capture semantics. Always compare against random baselines.

3. **Matryoshka Embeddings**: Train once, truncate to any dimension at inference. The Matryoshka
   loss evaluates at multiple prefix dimensions, enabling flexible quality-cost tradeoffs.

4. **Binary and Scalar Quantization**: INT8 gives 4x compression with negligible quality loss.
   Binary gives 32x compression for coarse-stage retrieval. Both are essential for billion-scale
   systems.

5. **Fine-Tuning Embeddings**: MNRL loss with domain-specific pairs adapts embeddings to your use
   case. Always evaluate on held-out data and check for isotropy degradation.

These techniques are not isolated -- they compose. A production embedding pipeline might fine-tune
a Matryoshka model on domain data, whiten the output space, and serve int8-quantized embeddings
behind a binary first-stage filter.

---

**Next**: `build.ipynb` -- Put it all together by building a fine-tuned embedding model with
full evaluation and deployment pipeline.